# Class 10 - RNAseq Analysis Continued

Start by Installing and Importing the necessary packages. Note that we will need to set some specific settings to get everything to run properly in Colaboratory.




In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels.stats.multitest as smm
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols
import seaborn as sns
import csv
import os
import sys
import argparse
from time import time


Setting up some other Pandas and python presets

In [ ]:
import  scipy.signal.signaltools

def _centered(arr, newsize):
    # Return the center newsize portion of the array.
    newsize = np.asarray(newsize)
    currsize = np.array(arr.shape)
    startind = (currsize - newsize) // 2
    endind = startind + newsize
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

scipy.signal.signaltools._centered = _centered

In [ ]:
from scipy.signal.signaltools import _centered as trim_centered

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [ ]:
pd.set_option('display.precision', 2)
pd.set_option('display.max_columns',10)

# We're also going to tell Jupyter to use inline plotting instead of notebook plotting
# It basically means you don't have to use plt.show() in every cell
%matplotlib inline

# and this command will allow multiple outputs from the same cell, rather than just the last one run
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

### Importing the data.

In [ ]:
#importing the raw melanoma dataset
CountsData = pd.read_csv('~/LECTURE_MATERIALS/DataFiles/melanoma_CountsRaw.csv',index_col = 0)

counts = CountsData.loc[:,'A1BG':]
counts.head(12)

# Normalizing data

Similarly to our last lesson, we will create log2CPM and DESeq2 normalized datasets for use in our differential expression pipeline.

For simplicity, we will skip the filtering steps where low-abundance and low-variance genes were removed, but you should do this for your own data!

In [ ]:

# Log2CPM normalized data
total_reads = counts.sum(axis = 1)
total_reads_per_million=total_reads.divide(1e6)
log2_cpm = np.log2(counts.add(1).divide(total_reads_per_million, axis=0))




In [ ]:
#DESeq2 normalized data
# Metadata data frame - the first few columns of the original counts csv file
meta = CountsData.iloc[:,0:5]

#make sure the number of sample rows match between counts and metadata
meta.shape
counts.shape

#Start by initializing the data, using the DeseqDataSet() function. The function requires 3 inputs:
# the CountsData data frame (containing the gene expression data in counts format),
# the meta data frame (containing the metadata associated with each sample),
# and the name of the column in meta that contains the group information for which comparison you want to make.
# To compare normal vs. metastatic, use "Stage"
dds = DeseqDataSet(counts = counts, metadata = meta, design_factors="Stage")

#Run DESeq2 normalization
dds.fit_size_factors()
dds.fit_genewise_dispersions()

dds



# Identifying Differentially Expressed Genes (DEGs)

## Introduction to linear models

Linear models allow for association testing between more than two variables (dimensions) and also have the added benefit of assessing the magnitude of change that the associated variables have.

These models attempt to fit straight lines through multi-dimensional space, and the magnitude of the slope in each dimension, as well as the goodness of fit, indicate extent of association.

To illustrate what linear models are doing, we'll start with some toy data that we'll generate.

We're going to make some data points that follow the general trend of the equation: y = 3x - 5, but will introduce some noise, so that the data points won't line up perfectly.

In [ ]:
# We will start by setting a random seed so that all our random variables match
np.random.seed(42)

In [ ]:
# create an array with 20 elements that have even spacing between -5 and 5.
x = np.linspace(-5, 5, 20)

# normal distributed noise around the equation y = 3x - 5
y = 3*x - 5 + 2 * np.random.normal(size=x.shape)

# Create a data frame containing all the relevant variables
data = pd.DataFrame({'x': x, 'y': y})
data.head()

**Visualizing the data**:
We'll discuss in more detail plotting later, but here's a brief view of the data we generated.

In [ ]:
# Plot the data
g = plt.figure(figsize=(5, 4))
g2 = plt.plot(x, y, 'o')

Our goal with the linear model will be to fit a line and determine the relationship between x and y based on the noisy data points generated.

In [ ]:
model = ols("y ~ x", data).fit()
print(model.summary())

Interpreting the Table:

First we have what’s the dependent variable and the model and the method. OLS stands for Ordinary Least Squares and the method “Least Squares” means that we’re trying to fit a regression line that would minimize the square of distance from the regression line. Date and Time are pretty self-explanatory :) So as number of observations.

Df of residuals and models relates to the degrees of freedom — “the number of values in the final calculation of a statistic that are free to vary.”

The coefficient of x ~3 means that as the x variable increases by 1, the predicted value of y increases by ~3.

The R-squared — the percentage of variance our model explains

The standard error - the standard deviation of the sampling distribution of a statistic, most commonly of the mean

The t scores and p-values report the statistics for hypothesis test — the x has statistically significant p-value (indicating significant association with y); there is a 95% confidence intervals for x (meaning we predict at a 95% percent confidence that the value of x is between 2.678 to 3.569).



We can also use the model parameters to plot the best fit line on the data

In [ ]:
# Plotting the regression output:
gB = plt.figure(figsize=(5, 4))
gB2 = plt.plot(x, y, 'o')

# creating a new set of ydata in the variable yline that based on the regression equation parameters calculated from ols()
yline = model.params[0] + model.params[1]*x
gB3 = plt.plot(x, yline, 'red')

## Simple DEG identification using linear models

We can identify differentially expressed genes between two groups of samples using linear models. Under the null hypothesis, the mean of the expression of the gene will not differ between two groups. A linear model determines the best fit relationship between gene expression and group membership, and by assigning a p-value to the strength of this relationship, we can decide whether to accept or reject this null hypothesis.


We will illustrate this by testing whether the gene PMEL varies between primary melanocytes and metastatic cells.

You might notice that group membership is not a continuous variable, like the examples we have looked at above, but consists of two categories. To account for this, binary categories can be encoded as '0's and '1's in the model, where '0' usually represents the 'control' category, and '1' the experimental 'treatment' category. Interpreting the final linear model, the intercept will be equivalent to the mean gene expression of the 'control' group (x=0), and the slope will estimate the effect of the 'treatment' on gene expression.

The form of the model we will test is

gene_expression ~ Stage

equivalent to the standard linear model formula, where y = gene_expression and x = Stage. The intercept does not need to be explicitly included.

y ~ m * x + b


A good first step to understand the differences is by plotting the expression values. Using dot plots we can see that the expression of PMEL varies strongly by cell line, and is lower in metastatic cell lines than primary melanocytes. We can test the strength of this association using a linear model.

Printing the summary of the model fit tells us a lot of information:

Note the slope and intercept, and compare their values to the mean gene expression in both groups. Here, the ols function has selected 'primary melanocytes' as the 'treatment' group.

In [ ]:
test_gene = 'PMEL'

pmel_expr = log2_cpm.loc[:,[test_gene]]
pmel_expr['Stage'] = CountsData['Stage']
pmel_expr['cell_line'] = CountsData['cell_line']

pmel_expr
plt.scatter(pmel_expr['cell_line'],pmel_expr[test_gene],color = 'green')
plt.show()
plt.scatter(pmel_expr['Stage'],pmel_expr[test_gene],color = 'red')

#C() tells python to treat this as a categorical variable while fitting the model
model_PMEL_Stage = ols("PMEL ~ C(Stage)", pmel_expr).fit()


print(model_PMEL_Stage.summary())

You can output the p-values quantifying whether each of the coefficients is significantly different from 0. If the p-value is significant for the coefficient associated with the "Stage" category, this suggests that there's a significant association between the expression of PMEL and the different Stage phenotypes.

In [ ]:
model_PMEL_Stage.pvalues
model_PMEL_Stage.pvalues['C(Stage)[T.primary melanocytes]']

### <font color=brown>Hands on practice</font>
Use `ols().fit()` to calculate the association between the gene PMEL and the different cell line types.




In [ ]:
### Calculating linear model for PMEL relative to cell line

#C() tells python to treat this as a categorical variable while fitting the model
model_PMEL_cell_line = 


print(model_PMEL_cell_line.summary())

Can you extend this code to perform linear modeling for each of the `top10VarGens`?

In [ ]:
top10VarGens = ['PMEL', 'TYRP1', 'AEBP1', 'GLUL', 'TYR', 'EEF1A2', 'CDC42EP1', 'A2M', 'SOD3', 'TGFBI']

In [ ]:
### extending linear modelling to all genes


## Using PyDESeq2 to identify DEGs

Behind the scenes, PyDESeq2 does comprehensively normalized the data and fits models. It can also attempt to more conservatively estimate 'shrunk' DEG estimates of fold changes, using a bayesian approach with a prior distribution, which helps prevent low-expression high-variability genes having large fold change estimates due to chance, and reduces the need for pre-filtering low expression genes. But we will not go into that here, and the fundamental principles DESeq2 uses are based on linear models as shown above.

Fitting linear models to every gene with DESeq2 just requires calling the .deseq2() method. Easy!

In [ ]:
# Run the DeSeq2 model fit.
dds.deseq2()

In [ ]:
dds

In [ ]:
dds.varm["LFC"]

In [ ]:
# Then calculate the statistics
deseq_stats = DeseqStats(dds,alpha = 0.05, contrast=["Stage", 'metastatic', 'primary melanocytes'])
deseq_stats.run_wald_test()
deseq_stats.lfc_shrink(coeff='Stage[T.primary melanocytes]')

deseq_stats.summary()

In [ ]:
# Constraining the results by adjusted p-value and log fold-change
sig_results = deseq_stats.results_df[(deseq_stats.results_df['padj'] < 0.05) &
			(abs(deseq_stats.results_df['log2FoldChange']) > 1)]
sig_results

In [ ]:
#sorting results
sig_results.sort_values('padj')

## Visualizing results with a volcano plot

A volcano plot is a scatter plot with fold changes plotted on the x-axis and -log10(p_adj) on the y axis. This allows us to quickly visualize the number and magnitude of differential expression, with the most changed genes at the peak of the 'eruption'


In [ ]:

top10_results = sig_results.sort_values('padj').head(10)
top10_results


plt.scatter(deseq_stats.results_df['log2FoldChange'],
            -np.log10(deseq_stats.results_df['padj']),
            color="grey")
plt.scatter(sig_results['log2FoldChange'],
            -np.log10(sig_results['padj']),
            color="orange")

plt.scatter(top10_results['log2FoldChange'],
            -np.log10(top10_results['padj']),
            color='red')

for index, row in top10_results.iterrows():
  plt.text(row['log2FoldChange'],-np.log10(row['padj']),index)

plt.xlabel("Log2 Fold Change")
plt.ylabel("-log10 p-adj")
plt.title("Metastatic vs primary melanocytes")


Now, let's save all the DESeq2 DEG results for next time...

In [ ]:
res_file = "deseq_results_melanoma.csv"
deseq_stats.results_df.to_csv(res_file)